In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from meta import *

df = pd.read_csv(savepath + r"/student_eda.csv")

df.info()

df['Pass_Fail'] = (df['Exam_Score'] >= 70).astype(int)

print("Pass/Fail Dağılımı:")
print(df['Pass_Fail'].value_counts())
print("\nOranlar:")
print(df['Pass_Fail'].value_counts(normalize=True) * 100)

X = df.drop(['Exam_Score', 'Pass_Fail'], axis=1)
y = df['Pass_Fail']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nÖzelliklerin listesi:\n{X.columns.tolist()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6378 entries, 0 to 6377
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Hours_Studied               6378 non-null   int64
 1   Attendance                  6378 non-null   int64
 2   Parental_Involvement        6378 non-null   int64
 3   Access_to_Resources         6378 non-null   int64
 4   Extracurricular_Activities  6378 non-null   int64
 5   Sleep_Hours                 6378 non-null   int64
 6   Previous_Scores             6378 non-null   int64
 7   Motivation_Level            6378 non-null   int64
 8   Internet_Access             6378 non-null   int64
 9   Tutoring_Sessions           6378 non-null   int64
 10  Family_Income               6378 non-null   int64
 11  Teacher_Quality             6378 non-null   int64
 12  School_Type                 6378 non-null   int64
 13  Peer_Influence              6378 non-null   int64
 14  Physical

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Eğitim seti boyutu: {X_train.shape}")
print(f"Test seti boyutu: {X_test.shape}")
print(f"\nEğitim setinde Pass/Fail oranı:\n{y_train.value_counts()}")
print(f"\nTest setinde Pass/Fail oranı:\n{y_test.value_counts()}")

Eğitim seti boyutu: (5102, 19)
Test seti boyutu: (1276, 19)

Eğitim setinde Pass/Fail oranı:
Pass_Fail
0    3837
1    1265
Name: count, dtype: int64

Test setinde Pass/Fail oranı:
Pass_Fail
0    960
1    316
Name: count, dtype: int64


In [3]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print("\n" + "=" * 60)
    print(f"{name} MODELİ")
    print("=" * 60)
    print(f"Doğruluk (Accuracy):    {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Kesinlik (Precision):   {precision:.4f} ({precision*100:.2f}%)")
    print(f"Duyarlılık (Recall):    {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1 Skoru:               {f1:.4f}")
    print("\nKARIŞIKLIK MATRİSİ")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print("\nSINIFLANDIRMA RAPORU")
    print(classification_report(y_test, y_pred, target_names=['Kaldı (0)', 'Geçti (1)']))

    return model, {
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    }

In [4]:
# AdaBoost
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
    n_estimators=100,
    random_state=42
)
ada_model, ada_metrics = evaluate_model('AdaBoost', ada_model, X_train, y_train, X_test, y_test)


AdaBoost MODELİ
Doğruluk (Accuracy):    0.9373 (93.73%)
Kesinlik (Precision):   0.9245 (92.45%)
Duyarlılık (Recall):    0.8133 (81.33%)
F1 Skoru:               0.8653

KARIŞIKLIK MATRİSİ
[[939  21]
 [ 59 257]]

SINIFLANDIRMA RAPORU
              precision    recall  f1-score   support

   Kaldı (0)       0.94      0.98      0.96       960
   Geçti (1)       0.92      0.81      0.87       316

    accuracy                           0.94      1276
   macro avg       0.93      0.90      0.91      1276
weighted avg       0.94      0.94      0.94      1276



In [5]:
# XGBoost
try:
    from xgboost import XGBClassifier

    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    )
    xgb_model, xgb_metrics = evaluate_model('XGBoost', xgb_model, X_train, y_train, X_test, y_test)
except Exception as e:
    print(f"\nXGBoost kurulumu/çalıştırılması sırasında hata: {e}")
    xgb_model = None
    xgb_metrics = {'Model': 'XGBoost', 'Accuracy': np.nan, 'Precision': np.nan, 'Recall': np.nan, 'F1': np.nan}


XGBoost MODELİ
Doğruluk (Accuracy):    0.9436 (94.36%)
Kesinlik (Precision):   0.9266 (92.66%)
Duyarlılık (Recall):    0.8386 (83.86%)
F1 Skoru:               0.8804

KARIŞIKLIK MATRİSİ
[[939  21]
 [ 51 265]]

SINIFLANDIRMA RAPORU
              precision    recall  f1-score   support

   Kaldı (0)       0.95      0.98      0.96       960
   Geçti (1)       0.93      0.84      0.88       316

    accuracy                           0.94      1276
   macro avg       0.94      0.91      0.92      1276
weighted avg       0.94      0.94      0.94      1276



c:\Users\karti\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [11:43:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [6]:
# LightGBM
try:
    from lightgbm import LGBMClassifier

    lgb_model = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
    lgb_model, lgb_metrics = evaluate_model('LightGBM', lgb_model, X_train, y_train, X_test, y_test)
except Exception as e:
    print(f"\nLightGBM kurulumu/çalıştırılması sırasında hata: {e}")
    lgb_model = None
    lgb_metrics = {'Model': 'LightGBM', 'Accuracy': np.nan, 'Precision': np.nan, 'Recall': np.nan, 'F1': np.nan}

[LightGBM] [Info] Number of positive: 1265, number of negative: 3837
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000594 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 191
[LightGBM] [Info] Number of data points in the train set: 5102, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.247942 -> initscore=-1.109619
[LightGBM] [Info] Start training from score -1.109619
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

In [7]:
results_df = pd.DataFrame([ada_metrics, xgb_metrics, lgb_metrics])
print("\n" + "=" * 60)
print("MODEL KARŞILAŞTIRMASI")
print("=" * 60)
print(results_df.sort_values('Accuracy', ascending=False).to_string(index=False))

# Tüm veri üzerinde tahmin yapma
models = {}
models['AdaBoost'] = ada_model
if xgb_model is not None:
    models['XGBoost'] = xgb_model
if lgb_model is not None:
    models['LightGBM'] = lgb_model

for name, model in models.items():
    df[f'{name}_Predicted_Pass_Fail'] = model.predict(X)
    if hasattr(model, 'predict_proba'):
        df[f'{name}_Pass_Probability'] = model.predict_proba(X)[:, 1]

print("\n" + "=" * 60)
print("TÜM VERİ İÇİN TAHMİN ÖZETİ")
print("=" * 60)
for name, model in models.items():
    pred_col = f'{name}_Predicted_Pass_Fail'
    pass_count = (df[pred_col] == 1).sum()
    fail_count = (df[pred_col] == 0).sum()
    total_count = len(df)
    print(f"{name}: Geçen={pass_count}, Kalan={fail_count}, Toplam={total_count}")


MODEL KARŞILAŞTIRMASI
   Model  Accuracy  Precision   Recall       F1
LightGBM  0.944357   0.935943 0.832278 0.881072
 XGBoost  0.943574   0.926573 0.838608 0.880399
AdaBoost  0.937304   0.924460 0.813291 0.865320

TÜM VERİ İÇİN TAHMİN ÖZETİ
AdaBoost: Geçen=1456, Kalan=4922, Toplam=6378
XGBoost: Geçen=1445, Kalan=4933, Toplam=6378
LightGBM: Geçen=1428, Kalan=4950, Toplam=6378
